In [3]:
import polars as pl
import pandas as pd
import json 
from pycomorb import comorbidity
import icu_flag
import hosps
import cim10_utils
import comorbidity_utils
import drug
import patient_utils

In [5]:
df = pl.read_parquet("final_patient_data.parquet")

In [6]:
def disease_medication_summary(
    df: pl.DataFrame,
    disease_cols: list[str],
    med_cols: list[str]
) -> pl.DataFrame:
    """
    Create a summary table for diseases and medication use.

    For each disease, the function calculates:
    - number of patients with the disease
    - percentage of all patients with the disease
    - percentage of patients with the disease who take each medication
    """

    # Total number of patients
    total_patients = df.height

    # Empty list to store rows
    result_rows = []

    # Loop over each disease
    for disease in disease_cols:

        # Keep only patients who have this disease
        patients_with_disease = df.filter(
            pl.col(disease) == 1
        )

        # Number of patients with this disease
        n_disease = patients_with_disease.height

        # Percentage of all patients with this disease
        percent_disease = n_disease / total_patients * 100

        # Create one row for this disease
        row = {
            "Disease": disease,
            "Number of patients": n_disease,
            "Percent of all patients": percent_disease
        }

        # Loop over each medication
        for med in med_cols:

            # Number of patients with this disease who also take this medication
            n_med = patients_with_disease.filter(
                pl.col(med) == 1
            ).height

            # Percentage among patients with this disease
            if n_disease > 0:
                percent_med = n_med / n_disease * 100
            else:
                percent_med = 0

            # Add this percentage to the row
            row[f"Percent taking {med}"] = percent_med

        # Add this row to the final result
        result_rows.append(row)

    # Convert result to Polars DataFrame
    result = pl.DataFrame(result_rows)

    return result

In [7]:
disease_cols = ["Cardiopathie ischémique", "Fibrillation atriale", "Insuffisance cardiaque chronique",
               "Pacemaker", "Pontage aorto-coronarien", "Insuffisance rénale chronique", "Antécédent d'AVC",
               "Cirrhose", "Cancer actif", "Immunodépression", "Dyslipidémie", "Diabète"]

In [8]:
drug_cols = ["beta_blocker", "ACEI", "ARB",  "anti_aldosterone",  "furosemide"]

In [10]:
table_disease_drug = disease_medication_summary(
    df = df,
    disease_cols= disease_cols,
    med_cols= drug_cols
)

In [11]:
table_disease_drug

Disease,Number of patients,Percent of all patients,Percent taking beta_blocker,Percent taking ACEI,Percent taking ARB,Percent taking anti_aldosterone,Percent taking furosemide
str,i64,f64,f64,f64,f64,f64,f64
"""Cardiopathie ischémique""",2,20.0,100.0,50.0,0.0,0.0,0.0
"""Fibrillation atriale""",2,20.0,50.0,50.0,0.0,0.0,0.0
"""Insuffisance cardiaque chroniq…",1,10.0,100.0,0.0,0.0,100.0,100.0
"""Pacemaker""",0,0.0,0.0,0.0,0.0,0.0,0.0
"""Pontage aorto-coronarien""",0,0.0,0.0,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…
"""Cirrhose""",0,0.0,0.0,0.0,0.0,0.0,0.0
"""Cancer actif""",1,10.0,0.0,0.0,0.0,0.0,0.0
"""Immunodépression""",0,0.0,0.0,0.0,0.0,0.0,0.0
